# SENSITIVITY_PassC — supplementary sensitivity and construct-validation checks around the frozen final analysis

**Status: supplementary.** The frozen subgroup definitions and every result reported in the thesis remain primary. This notebook only asks whether the conclusions are reasonably stable under plausible alternatives. It never refits a model.

Issues covered (supervisory tracker): **#22** reference-group composition (first), **#18** emoji threshold, **#19** slang threshold and lexicon, **#21** precedence order, **#30/#31** direct gap intervals and resample stability, **#51** Sentiment140 emoji gap.

Guards: before any sensitivity result is computed, the notebook (1) reproduces the frozen subgroup assignment row for row, (2) reproduces the locked subgroup Macro F1 values, and (3) re-scores Sentiment140 with the locked pipeline and reproduces the locked Macro F1, checking both SHA-256 fingerprints. Any mismatch stops the run.

Inputs (read-only): `LJMU_FINAL_EXPERIMENT_RUN/predictions/FINAL_SELECTED_TweetEval.parquet`, the locked pipeline, the frozen Sentiment140 sample, `final_thesis_utils.py`. Outputs: `LJMU_FINAL_EXPERIMENT_RUN/results/sensitivity/`.

In [1]:
from google.colab import drive; drive.mount('/content/drive')
!pip -q install emoji vaderSentiment
import io, os, re, glob, json, hashlib, pickle, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score
ROOT = Path('/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN')
OUT = ROOT/'results'/'sensitivity'; OUT.mkdir(parents=True, exist_ok=True)
SEED = 42; B_MAIN = 2000
# load the frozen utility module exactly as the FINAL notebooks do
import sys; sys.path.insert(0, '/content')
try:
    import final_thesis_utils as ftu
except ImportError:
    import google.auth
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user(); creds, _ = google.auth.default()
    req = build('drive', 'v3', credentials=creds, cache_discovery=False).files().get_media(fileId='14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02')
    with io.FileIO('/content/final_thesis_utils.py', 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req); done = False
        while not done: _, done = dl.next_chunk()
    import final_thesis_utils as ftu
REF = getattr(ftu, 'REFERENCE_GROUP', 'reference-unmarked')
FOCAL = ['emoji-heavy', 'slang-heavy', 'sarcasm-indicated']
def macro_f1(y, p, labels=None):
    return f1_score(y, p, average='macro', labels=labels, zero_division=0)
def save(df, name):
    df.to_csv(OUT/name, index=False); print('saved', name); return df
print('utils loaded; reference group =', REF)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.1 MB/s eta 0:00:00


utils loaded; reference group = reference-unmarked


## 0. Guards: reproduce the frozen assignment and locked results

In [2]:
pred = pd.read_parquet(ROOT/'predictions'/'FINAL_SELECTED_TweetEval.parquet')
assert len(pred) == 12284, len(pred)
tagged = ftu.tag_subgroups(pred[['text']].copy(), 'text')
cols = list(tagged.columns); print('columns produced by tag_subgroups:', cols)
pick = lambda *ws: next((c for c in cols if all(w in c.lower() for w in ws)), None)
WC, EC, SC = pick('word', 'count'), pick('emoji', 'count'), pick('slang', 'count')
SF = next((c for c in cols if 'sarcas' in c.lower() and c != 'subgroup'), None)
print('word count:', WC, '| emoji count:', EC, '| slang count:', SC, '| sarcasm flag:', SF)
assert all([WC, EC, SC, SF]), 'tagger columns not found: set WC/EC/SC/SF by hand from the list above'
F = pd.DataFrame({'wc': tagged[WC].astype(float).values, 'ec': tagged[EC].astype(float).values,
                  'sc': tagged[SC].astype(float).values, 'sarc': tagged[SF].astype(bool).values})
F['y'] = pred['y_true'].values; F['p'] = pred['y_pred'].values; F['subgroup'] = pred['subgroup'].values
F['words'] = pred['text'].astype(str).str.split().str.len().values

def assign(df, e_thr=0.05, s_thr=0.10, order=('sarcasm', 'emoji', 'slang'), min_words=8):
    wc = df.wc.replace(0, np.nan)
    flags = {'sarcasm': df.sarc.values, 'emoji': (df.ec / wc > e_thr).fillna(False).values,
             'slang': (df.sc / wc > s_thr).fillna(False).values}
    name = {'sarcasm': 'sarcasm-indicated', 'emoji': 'emoji-heavy', 'slang': 'slang-heavy'}
    out = np.where(df.wc.values >= min_words, REF, 'other').astype(object)
    for k in reversed(order):                      # highest precedence applied last
        out = np.where(flags[k], name[k], out)
    return out

frozen = assign(F)
agree = (frozen == F.subgroup.values).mean()
print(f'frozen-rule reproduction: {agree:.6f} of rows match the locked subgroup labels')
assert agree == 1.0, 'reconstructed rule does not reproduce the frozen assignment exactly; stop and inspect'
LOCKED = {'reference-unmarked': 0.596926, 'emoji-heavy': 0.635388, 'slang-heavy': 0.619431, 'sarcasm-indicated': 0.789744}
for g, v in LOCKED.items():
    m = F.subgroup == g; got = macro_f1(F.y[m], F.p[m])
    print(f'{g:20s} locked {v:.6f}  reproduced {got:.6f}'); assert abs(got - v) < 1e-6
print('Guards passed: assignment and locked subgroup Macro F1 reproduced exactly.')

columns produced by tag_subgroups: ['text', 'word_count', 'emoji_count', 'emoji_density', 'slang_count', 'slang_ratio', 'sarcasm_indicator', 'punct_intensity', 'hashtag_count', 'negation_count', 'is_emoji_heavy', 'is_slang_heavy', 'is_sarcasm_indicated', 'is_reference_candidate', 'subgroup']
word count: word_count | emoji count: emoji_count | slang count: slang_count | sarcasm flag: sarcasm_indicator
frozen-rule reproduction: 1.000000 of rows match the locked subgroup labels
reference-unmarked   locked 0.596926  reproduced 0.596926
emoji-heavy          locked 0.635388  reproduced 0.635388
slang-heavy          locked 0.619431  reproduced 0.619431
sarcasm-indicated    locked 0.789744  reproduced 0.789744
Guards passed: assignment and locked subgroup Macro F1 reproduced exactly.


## 1. #22 Reference-group composition and standardised comparisons

In [3]:
rows = []
for g in [REF] + FOCAL + ['other']:
    d = F[F.subgroup == g]
    sh = d.y.value_counts(normalize=True).reindex([0, 1, 2], fill_value=0)
    rows.append({'Subgroup': g, 'N': len(d), 'Mean words': round(d.words.mean(), 2), 'Median words': d.words.median(),
                 'Share <8 words': round((d.words < 8).mean(), 4),
                 'Negative share': round(sh[0], 4), 'Neutral share': round(sh[1], 4), 'Positive share': round(sh[2], 4)})
comp = save(pd.DataFrame(rows), 'C22_composition.csv'); display(comp)

def class_acc(d): return {c: (d.p[d.y == c] == c).mean() if (d.y == c).any() else np.nan for c in (0, 1, 2)}
ref = F[F.subgroup == REF]; w_ref = ref.y.value_counts(normalize=True).reindex([0, 1, 2]).values
acc_ref = (ref.p == ref.y).mean()
std = []
for g in FOCAL:
    d = F[F.subgroup == g]; ca = class_acc(d)
    raw = (d.p == d.y).mean() - acc_ref
    stdv = np.nansum([w_ref[c] * ca[c] for c in (0, 1, 2)]) / np.nansum([w_ref[c] for c in (0, 1, 2) if not np.isnan(ca[c])]) - acc_ref
    std.append({'Subgroup': g, 'N': len(d), 'Raw CPD': round(raw, 4), 'Class-standardised CPD (reference class mix)': round(stdv, 4)})
std = save(pd.DataFrame(std), 'C22_class_standardised_cpd.csv'); display(std)

# length-matched reference: reweight reference posts to each focal group's word-count distribution
bins = [0, 8, 12, 16, 20, 25, 30, 100]
F['lbin'] = pd.cut(F.words, bins, right=False)
lm = []
for g in FOCAL:
    d = F[F.subgroup == g]; tgt = d.lbin.value_counts(normalize=True)
    r = F[F.subgroup == REF]; src = r.lbin.value_counts(normalize=True)
    wts = r.lbin.map(lambda b: tgt.get(b, 0) / src.get(b, np.nan) if src.get(b, 0) > 0 else 0).fillna(0).astype(float)
    acc_lm = np.average((r.p == r.y).astype(float), weights=wts) if wts.sum() > 0 else np.nan
    lm.append({'Subgroup': g, 'Focal accuracy': round((d.p == d.y).mean(), 4), 'Reference accuracy (raw)': round(acc_ref, 4),
               'Reference accuracy (length-matched)': round(acc_lm, 4),
               'Focal share of posts in bins absent from reference': round(d.lbin.map(lambda b: src.get(b, 0) == 0).mean(), 4)})
lm = save(pd.DataFrame(lm), 'C22_length_matched_reference.csv'); display(lm)

# alternative reference: drop the minimum-length rule (unmarked posts of any length)
alt = np.where(F.subgroup.isin([REF, 'other']), REF, F.subgroup)
ar = []
for name, labels in (('Frozen (>= 8 words)', F.subgroup.values), ('No minimum length', alt)):
    rm = labels == REF; rf1 = macro_f1(F.y[rm], F.p[rm])
    for g in FOCAL:
        m = labels == g
        ar.append({'Reference definition': name, 'Reference N': int(rm.sum()), 'Subgroup': g, 'Macro F1': round(macro_f1(F.y[m], F.p[m]), 4),
                   'Reference Macro F1': round(rf1, 4), 'Gap': round(macro_f1(F.y[m], F.p[m]) - rf1, 4)})
ar = save(pd.DataFrame(ar), 'C22_alternative_reference.csv'); display(ar)

saved C22_composition.csv


,Subgroup,N,Mean words,Median words,Share <8 words,Negative share,Neutral share,Positive share
0,reference-unmarked,10388,16.04,16.0,0.0000,0.3472,0.4822,0.1706
1,emoji-heavy,710,12.60,12.0,0.1859,0.2113,0.3423,0.4465
2,slang-heavy,60,8.85,8.0,0.4500,0.2833,0.4333,0.2833
3,sarcasm-indicated,14,15.64,17.0,0.1429,0.3571,0.5000,0.1429
4,other,1112,5.58,6.0,1.0000,0.1736,0.5863,0.2401


saved C22_class_standardised_cpd.csv


,Subgroup,N,Raw CPD,Class-standardised CPD (reference class mix)
0,emoji-heavy,710,0.0374,-0.0088
1,slang-heavy,60,0.0019,-0.0289
2,sarcasm-indicated,14,0.1710,0.1780


saved C22_length_matched_reference.csv


,Subgroup,Focal accuracy,Reference accuracy (raw),Reference accuracy (length-matched),Focal share of posts in bins absent from reference
0,emoji-heavy,0.6521,0.6147,0.6243,0.1859
1,slang-heavy,0.6167,0.6147,0.6408,0.4500
2,sarcasm-indicated,0.7857,0.6147,0.6088,0.1429


saved C22_alternative_reference.csv


,Reference definition,Reference N,Subgroup,Macro F1,Reference Macro F1,Gap
0,Frozen (>= 8 words),10388,emoji-heavy,0.6354,0.5969,0.0385
1,Frozen (>= 8 words),10388,slang-heavy,0.6194,0.5969,0.0225
2,Frozen (>= 8 words),10388,sarcasm-indicated,0.7897,0.5969,0.1928
3,No minimum length,11500,emoji-heavy,0.6354,0.5991,0.0362
4,No minimum length,11500,slang-heavy,0.6194,0.5991,0.0203
5,No minimum length,11500,sarcasm-indicated,0.7897,0.5991,0.1906


## 2. #18 Emoji threshold, #19 slang threshold and lexicon, #21 precedence order

In [4]:
def gaps(labels, tag):
    rm = labels == REF; rf1 = macro_f1(F.y[rm], F.p[rm]); out = []
    for g in FOCAL:
        m = labels == g
        out.append({'Setting': tag, 'Subgroup': g, 'N': int(m.sum()),
                    'Macro F1': round(macro_f1(F.y[m], F.p[m]), 4) if m.any() else np.nan,
                    'Gap vs reference': round(macro_f1(F.y[m], F.p[m]) - rf1, 4) if m.any() else np.nan})
    return out
res = []
for t in (0.03, 0.05, 0.075, 0.10): res += gaps(assign(F, e_thr=t), f'emoji threshold {t}' + (' (frozen)' if t == 0.05 else ''))
res = save(pd.DataFrame(res), 'C18_emoji_threshold.csv'); display(res[res.Subgroup == 'emoji-heavy'])
res = []
for t in (0.05, 0.075, 0.10, 0.15): res += gaps(assign(F, s_thr=t), f'slang threshold {t}' + (' (frozen)' if t == 0.10 else ''))
res = save(pd.DataFrame(res), 'C19_slang_threshold.csv'); display(res[res.Subgroup == 'slang-heavy'])
lex = getattr(ftu, 'SLANG_TERMS', None) or getattr(ftu, 'SLANG_LEXICON', None)
if lex is not None:
    save(pd.DataFrame({'term': sorted(lex)}), 'C19_slang_lexicon.csv'); print(len(lex), 'lexicon terms written')
else:
    print('slang lexicon constant not found by name in final_thesis_utils; list it from the module manually')
# #21: overlap before precedence, and alternative orders
e = (F.ec / F.wc.replace(0, np.nan) > 0.05).fillna(False); s = (F.sc / F.wc.replace(0, np.nan) > 0.10).fillna(False); c = F.sarc
ov = pd.DataFrame({'Flags': ['emoji only', 'slang only', 'sarcasm only', 'emoji+slang', 'emoji+sarcasm', 'slang+sarcasm', 'all three'],
                   'Posts': [int((e & ~s & ~c).sum()), int((~e & s & ~c).sum()), int((~e & ~s & c).sum()), int((e & s & ~c).sum()),
                             int((e & ~s & c).sum()), int((~e & s & c).sum()), int((e & s & c).sum())]})
save(ov, 'C21_flag_overlap.csv'); display(ov)
res = []
for order in (('sarcasm', 'emoji', 'slang'), ('emoji', 'sarcasm', 'slang'), ('slang', 'emoji', 'sarcasm'), ('emoji', 'slang', 'sarcasm')):
    lab = assign(F, order=order); res += gaps(lab, ' > '.join(order) + (' (frozen)' if order == ('sarcasm', 'emoji', 'slang') else ''))
res = save(pd.DataFrame(res), 'C21_precedence_order.csv'); display(res)

saved C18_emoji_threshold.csv


,Setting,Subgroup,N,Macro F1,Gap vs reference
0,emoji threshold 0.03,emoji-heavy,808,0.6444,0.0492
3,emoji threshold 0.05 (frozen),emoji-heavy,710,0.6354,0.0385
6,emoji threshold 0.075,emoji-heavy,567,0.6337,0.0360
9,emoji threshold 0.1,emoji-heavy,444,0.6281,0.0292


saved C19_slang_threshold.csv


,Setting,Subgroup,N,Macro F1,Gap vs reference
1,slang threshold 0.05,slang-heavy,273,0.5823,-0.0147
4,slang threshold 0.075,slang-heavy,138,0.5713,-0.0258
7,slang threshold 0.1 (frozen),slang-heavy,60,0.6194,0.0225
10,slang threshold 0.15,slang-heavy,24,0.6289,0.0318


saved C19_slang_lexicon.csv
51 lexicon terms written
saved C21_flag_overlap.csv


,Flags,Posts
0,emoji only,700
1,slang only,60
2,sarcasm only,13
3,emoji+slang,10
4,emoji+sarcasm,1
5,slang+sarcasm,0
6,all three,0


saved C21_precedence_order.csv


,Setting,Subgroup,N,Macro F1,Gap vs reference
0,sarcasm > emoji > slang (frozen),emoji-heavy,710,0.6354,0.0385
1,sarcasm > emoji > slang (frozen),slang-heavy,60,0.6194,0.0225
2,sarcasm > emoji > slang (frozen),sarcasm-indicated,14,0.7897,0.1928
3,emoji > sarcasm > slang,emoji-heavy,711,0.6359,0.0390
4,emoji > sarcasm > slang,slang-heavy,60,0.6194,0.0225
5,emoji > sarcasm > slang,sarcasm-indicated,13,0.7758,0.1788
6,slang > emoji > sarcasm,emoji-heavy,701,0.6336,0.0367
7,slang > emoji > sarcasm,slang-heavy,70,0.6525,0.0555
8,slang > emoji > sarcasm,sarcasm-indicated,13,0.7758,0.1788
9,emoji > slang > sarcasm,emoji-heavy,711,0.6359,0.0390


## 3. #30/#31 Direct gap intervals and resample stability

In [5]:
def boot_gap(yg, pg, yr, pr, B, seed=SEED, labels=None):
    rng = np.random.default_rng(seed); ng, nr = len(yg), len(yr); vals = np.empty(B)
    for b in range(B):
        ig = rng.integers(0, ng, ng); ir = rng.integers(0, nr, nr)
        vals[b] = macro_f1(yg[ig], pg[ig], labels) - macro_f1(yr[ir], pr[ir], labels)
    return np.quantile(vals, [0.025, 0.975])
y, p = F.y.values, F.p.values; rm = (F.subgroup == REF).values
rows = []
for g in FOCAL:
    m = (F.subgroup == g).values; gap = macro_f1(y[m], p[m]) - macro_f1(y[rm], p[rm])
    for B in (500, B_MAIN):
        lo, hi = boot_gap(y[m], p[m], y[rm], p[rm], B)
        rows.append({'Subgroup': g, 'N': int(m.sum()), 'Gap': round(gap, 4), 'Resamples': B, 'CI low': round(lo, 4), 'CI high': round(hi, 4),
                     'Excludes zero': bool(lo > 0 or hi < 0)})
gp = save(pd.DataFrame(rows), 'C30_C31_gap_intervals_tweeteval.csv'); display(gp)

saved C30_C31_gap_intervals_tweeteval.csv


,Subgroup,N,Gap,Resamples,CI low,CI high,Excludes zero
0,emoji-heavy,710,0.0385,500,0.0021,0.0766,True
1,emoji-heavy,710,0.0385,2000,-0.0013,0.0752,False
2,slang-heavy,60,0.0225,500,-0.1043,0.1275,False
3,slang-heavy,60,0.0225,2000,-0.1113,0.1367,False
4,sarcasm-indicated,14,0.1928,500,-0.1597,0.3995,False
5,sarcasm-indicated,14,0.1928,2000,-0.1425,0.4006,False


## 4. #51 Sentiment140 emoji gap (locked pipeline re-scored; no refit)

In [6]:
def sha(path):
    h = hashlib.sha256();
    with open(path, 'rb') as fh:
        for ch in iter(lambda: fh.read(1 << 20), b''): h.update(ch)
    return h.hexdigest()
pipe_path = next(iter(glob.glob(str(ROOT/'**'/'FINAL_SELECTED_PIPELINE.pkl'), recursive=True)), None)
s140_path = next(iter(glob.glob('/content/drive/MyDrive/**/df_sentiment140_eval_sample_final.parquet', recursive=True)), None)
print('pipeline:', pipe_path); print('sample:', s140_path)
assert pipe_path and s140_path, 'locate the locked pipeline and the frozen Sentiment140 sample and set the paths by hand'
assert sha(pipe_path) == 'bf92c6162dd4664c4a3445d8ce806cc28b09809f75613d6c70f4ef8b6ad954e2', 'pipeline fingerprint mismatch'
assert sha(s140_path) == 'd25327ccb390945e945f687c38f84e3973e81199c7b7977f02d1f5a08593b8ed', 'Sentiment140 fingerprint mismatch'
with open(pipe_path, 'rb') as fh: pipe = pickle.load(fh)
s = pd.read_parquet(s140_path); assert len(s) == 100000
s['subgroup'] = ftu.tag_subgroups(s[['text']].copy(), 'text')['subgroup'].values
ys = s['sentiment'].str.lower().map({'negative': 0, 'positive': 2}).values
ps = np.asarray(pipe.predict(s['text_eval'].astype(str)))
f_all = macro_f1(ys, ps, labels=[0, 2]); print(f'Sentiment140 Macro F1 reproduced: {f_all:.6f} (locked 0.580373)')
assert abs(f_all - 0.580373) < 1e-5, 'locked Sentiment140 result not reproduced; stop'
r = (s.subgroup == REF).values; rows = []
for g in FOCAL:
    m = (s.subgroup == g).values
    if m.sum() == 0: continue
    gap = macro_f1(ys[m], ps[m], [0, 2]) - macro_f1(ys[r], ps[r], [0, 2])
    lo, hi = boot_gap(ys[m], ps[m], ys[r], ps[r], B_MAIN, labels=[0, 2])
    sup = pd.Series(ys[m]).value_counts().reindex([0, 2], fill_value=0)
    rows.append({'Subgroup': g, 'N': int(m.sum()), 'Negative N': int(sup[0]), 'Positive N': int(sup[2]),
                 'Macro F1': round(macro_f1(ys[m], ps[m], [0, 2]), 4), 'Gap': round(gap, 4), 'Resamples': B_MAIN,
                 'CI low': round(lo, 4), 'CI high': round(hi, 4), 'Excludes zero': bool(lo > 0 or hi < 0)})
s51 = save(pd.DataFrame(rows), 'C51_sentiment140_gap_intervals.csv'); display(s51)

pipeline: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN/models/FINAL_SELECTED_PIPELINE.pkl
sample: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/Sentiment140/df_sentiment140_eval_sample_final.parquet
Sentiment140 Macro F1 reproduced: 0.580373 (locked 0.580373)
saved C51_sentiment140_gap_intervals.csv


,Subgroup,N,Negative N,Positive N,Macro F1,Gap,Resamples,CI low,CI high,Excludes zero
0,emoji-heavy,72,27,45,0.3690,-0.2123,2000,-0.2941,-0.1140,True
1,slang-heavy,2795,1492,1303,0.6213,0.0400,2000,0.0203,0.0581,True
2,sarcasm-indicated,171,80,91,0.6233,0.0419,2000,-0.0379,0.1143,False


## 5. Samples for manual construct checks (Pass D) and run summary

In [7]:
smp = []
for g, n in (('emoji-heavy', 20), ('slang-heavy', 20), ('sarcasm-indicated', 14)):
    d = pred[F.subgroup.values == g]
    smp.append(d.sample(min(n, len(d)), random_state=SEED)[['row_id', 'text', 'true_label', 'pred_label']].assign(subgroup=g))
save(pd.concat(smp), 'C18_C20_manual_check_sample.csv')
summary = {'notebook': 'SENSITIVITY_PassC', 'status': 'supplementary; frozen primary results unchanged',
           'guards': {'assignment_reproduced': True, 'locked_subgroup_f1_reproduced': True, 'sentiment140_reproduced': True,
                      'sha256_checked': True}, 'resamples_main': B_MAIN, 'seed': SEED, 'outputs': sorted(p.name for p in OUT.glob('*.csv'))}
json.dump(summary, open(OUT/'PASSC_SUMMARY.json', 'w'), indent=2); print(json.dumps(summary, indent=2))

saved C18_C20_manual_check_sample.csv
{
  "notebook": "SENSITIVITY_PassC",
  "status": "supplementary; frozen primary results unchanged",
  "guards": {
    "assignment_reproduced": true,
    "locked_subgroup_f1_reproduced": true,
    "sentiment140_reproduced": true,
    "sha256_checked": true
  },
  "resamples_main": 2000,
  "seed": 42,
  "outputs": [
    "C18_C20_manual_check_sample.csv",
    "C18_emoji_threshold.csv",
    "C19_slang_lexicon.csv",
    "C19_slang_threshold.csv",
    "C21_flag_overlap.csv",
    "C21_precedence_order.csv",
    "C22_alternative_reference.csv",
    "C22_class_standardised_cpd.csv",
    "C22_composition.csv",
    "C22_length_matched_reference.csv",
    "C30_C31_gap_intervals_tweeteval.csv",
    "C51_sentiment140_gap_intervals.csv"
  ]
}
